# 12 · Forest and Vegetation Analysis

Monitor forests, estimate canopy height, detect deforestation.

## Contents
1. Deforestation detection
2. Canopy height estimation
3. Tree species classification
4. Biomass and carbon estimation
5. Forest fire detection

## 1 · Deforestation Detection

In [ ]:
import pygeovision as pgv

client = pgv.PyGeoVision()

# Amazon deforestation monitoring -- real DeforestationPipeline directly.
# Note the real parameter names are baseline_year/analysis_year, not
# date_before/date_after.
from pygeovision.ai.pipelines import DeforestationPipeline

pipeline = DeforestationPipeline(client)
result = pipeline.run(
    bbox=(-60.0, -3.0, -59.5, -2.5),    # Amazon, Brazil
    output_dir="./results/amazon/",
    baseline_year="2015-01",
    analysis_year="2024-01",
)

print(f"Deforestation pipeline: {'[OK]' if result.success else '[FAIL]'}")
if result.success:
    print(f"  Output: {result.output_path}")
    print("  Period: 2015 -> 2024")

## 2 · Canopy Height Estimation

In [ ]:
# Real CanopyHeightPipeline class directly (Tier-3-fixed, not yet
# independently re-verified against real data this cycle).
from pygeovision.ai.pipelines.domains import CanopyHeightPipeline

pipeline = CanopyHeightPipeline(client)
result = pipeline.run(
    bbox=(-122.4, 37.7, -122.3, 37.8),   # Redwood forests, CA
    output_dir="./results/canopy/",
    date="2024-07",
)

print(f"Canopy height pipeline: {'[OK]' if result.success else '[FAIL]'}")
if result.success:
    print(f"  Output: {result.output_path}")

# Real CHMv2Model direct call -- an earlier version of this cell used a
# fabricated client.geoai.canopy.estimate call.
from pygeovision.models.foundation.dinov3 import CHMv2Model

results = client.search(
    bbox=(-122.4, 37.7, -122.3, 37.8),
    date_range=("2024-06-01", "2024-08-31"),
    providers=["planetary_computer"], cloud_cover_max=5, max_results=2,
)
if results:
    dl = client.download(results[:1], output_dir="./data/redwood/",
                          post_process=["reproject:EPSG:4326", "cog"])
    if dl[0].success and dl[0].path:
        try:
            chm = CHMv2Model()
            chm.predict_canopy_height(
                dl[0].path,
                output_path="./results/canopy/canopy_height.tif",
            )
            print("  Direct canopy estimation complete")
        except Exception as e:
            print(f"  Canopy height model: {e}")

## 3 · Carbon Estimation

In [ ]:
import numpy as np

# Carbon estimation from NDVI + allometric relationships
# Based on: carbon_density (tC/ha) = f(NDVI, forest type)

# Simulated NDVI values for tropical forest pixels
np.random.seed(42)
n_pixels = 10000
ndvi = np.random.beta(6, 2, n_pixels) * 0.4 + 0.6   # Tropical forest NDVI

# Simplified allometric model (Brown 1997)
# AGB (tDW/ha) ≈ 52.27 × NDVI^3.16  (for tropical broadleaf)
agb = 52.27 * ndvi**3.16                               # Above-ground biomass tDW/ha
carbon = agb * 0.47                                    # ~47% of dry weight is carbon

pixel_area_ha = 0.01                                   # 10m Sentinel-2 pixels

total_agb_t = agb.sum() * pixel_area_ha
total_carbon_t = carbon.sum() * pixel_area_ha
total_co2_t = total_carbon_t * 3.67                    # C -> CO2

print("Carbon Stock Estimation — Tropical Forest Region")
print(f"{'='*55}")
print(f"  Area analysed:         {n_pixels * pixel_area_ha:.0f} ha")
print(f"  Mean NDVI:             {ndvi.mean():.3f}")
print(f"  Mean AGB:              {agb.mean():.1f} tDW/ha")
print(f"  Total AGB:             {total_agb_t/1000:.1f} kilotonnes DW")
print(f"  Total carbon:          {total_carbon_t:.0f} tonnes C")
print(f"  CO2 equivalent:        {total_co2_t:.0f} tonnes CO2")
print(f"  Carbon density:        {carbon.mean():.1f} tC/ha")

# Carbon value (at $50/tCO2 — voluntary carbon markets)
carbon_value = total_co2_t * 50
print(f"  Carbon value (@$50/t): ${carbon_value:,.0f}")

Carbon Stock Estimation — Tropical Forest Region
  Area analysed:         100 ha
  Mean NDVI:             0.800
  Mean AGB:              249.3 tDW/ha
  Total AGB:             24.9 kilotonnes DW
  Total carbon:          11,718 tonnes C
  CO2 equivalent:        43,007 tonnes CO2
  Carbon density:        117.2 tC/ha
  Carbon value (@$50/t): $2,150,374


## 4 · Forest Fire Detection

In [ ]:
# Real ForestFirePipeline -- a real subclass of the audited
# WildfireSeverityPipeline (real dNBR burn-scar mapping). Requires
# date_before/date_after (a real, bi-temporal comparison), not a single
# date the way an earlier version of this cell used. Covers burn-scar
# mapping only -- not real-time active-fire detection.
from pygeovision.ai.pipelines.domains import ForestFirePipeline

pipeline = ForestFirePipeline(client)
result = pipeline.run(
    bbox=(-120.5, 38.5, -119.5, 39.2),   # Sierra Nevada, CA
    output_dir="./results/fire/",
    date_before="2024-05",
    date_after="2024-08",
)
print(f"Forest fire (burn scar) pipeline: {'[OK]' if result.success else '[FAIL]'}")
if result.success:
    print(f"  Output: {result.output_path}")
    print(f"  Stats: {result.stats}")

## 5 · Tree Species Classification

In [ ]:
# Real TreeSpeciesPipeline class directly. Honest limitation: this was
# fixed this cycle to correctly return success=False -- no trained
# tree-species classifier exists in this codebase; imagery is downloaded
# but not classified. An earlier version of this cell (and this
# pipeline) misleadingly returned success=True with no real work done.
from pygeovision.ai.pipelines.domains import TreeSpeciesPipeline

pipeline = TreeSpeciesPipeline(client)
result = pipeline.run(
    bbox=(-122.4, 37.7, -122.3, 37.8),
    output_dir="./results/tree_species/",
    date="2024-07",
)
print(f"Tree species pipeline: {'[OK]' if result.success else '[FAIL, honestly: ' + str(result.error) + ']'}")

# Available datasets for tree species classification -- pygeovision's
# real, own dataset registry (not "EarthNets", which an earlier version
# of this cell incorrectly called it).
from pygeovision.datasets.registry import dataset_registry

tree_datasets = dataset_registry.search("tree")
print(f"\nReal dataset registry -- tree-related entries ({len(tree_datasets)}):")
for d in tree_datasets[:5]:
    print(f"  {d.name:<30} {d.domain:<12} {d.modality}")